### 0. Install Dependencies

In [1]:
!pip install google-search-results pandas python-dotenv requests trafilatura youtube_transcript_api -q

### 1. Imports & Configuration

In [ ]:
from serpapi import GoogleSearch
import pandas as pd
from dotenv import load_dotenv
import os
import time

load_dotenv()
SERP_API_KEY = os.getenv("SERP_API_KEY")

# Max number of job results to collect per query
MAX_NUM_RESULTS = 10

### 2. SerpApi - Collecting Google Jobs results

In [10]:
def collect_google_jobs(
    queries: list[str],
    location: str = "United States",
    num_results: int = MAX_NUM_RESULTS
):
    """
    Search Google Jobs for each query and collect structured job posting metadata.

    Returns:
        jobs_df     : DataFrame of collected job metadata
        raw_results : list of raw SerpAPI responses for inspection
    """
    job_rows = []
    raw_results = []

    for query in queries:
        print(f"\n🔍 Searching Google Jobs for: '{query}'")

        params = {
            "engine": "google_jobs",
            "q": query,
            "location": location,
            "google_domain": "google.com",
            "hl": "en",
            "api_key": SERP_API_KEY
        }

        search = GoogleSearch(params)
        results = search.get_dict()

        raw_results.append({"query": query, "results": results})
        jobs_results = results.get("jobs_results", [])

        if not jobs_results:
            print(f"  ⚠️ No jobs_results for: {query}")
            continue

        for job in jobs_results[:num_results]:
            detected_extensions = job.get("detected_extensions", {}) or {}
            apply_options = job.get("apply_options", []) or []

            job_rows.append({
                "query": query,
                "title": job.get("title"),
                "company": job.get("company_name"),
                "location": job.get("location"),
                "via": job.get("via"),
                "description": job.get("description"),
                "thumbnail": job.get("thumbnail"),
                "job_id": job.get("job_id"),
                "posted_at": detected_extensions.get("posted_at"),
                "schedule_type": detected_extensions.get("schedule_type"),
                "salary": detected_extensions.get("salary"),
                "work_from_home": detected_extensions.get("work_from_home"),
                "apply_links": apply_options,
                "type": "job_posting"
            })

        print(f"  ✅ {min(len(jobs_results), num_results)} jobs collected")

    df = pd.DataFrame(job_rows)

    if df.empty:
        print("\n⚠️ No job postings collected across all queries.")
        return df, raw_results

    # Remove duplicates by job_id when available
    if "job_id" in df.columns and df["job_id"].notna().any():
        df = df.drop_duplicates(subset="job_id").reset_index(drop=True)
    else:
        df = df.drop_duplicates(subset=["title", "company", "location"]).reset_index(drop=True)

    return df, raw_results


job_queries = [
    "Machine Learning Engineer",
    "Data Engineer",
    "MLOps Engineer",
    "AI Engineer",
    "Data Scientist",
    "Applied Scientist",
    "NLP Engineer",
    "Computer Vision Engineer",
    "Deep Learning Engineer",
    "LLM Engineer",
    "Software Engineer Machine Learning",
    "ML Platform Engineer",
]

df_jobs, raw_jobs = collect_google_jobs(
    queries=job_queries,
    location="United States",
    num_results=10
)

df_jobs.head()


🔍 Searching Google Jobs for: 'Machine Learning Engineer'
  ✅ 10 jobs collected

🔍 Searching Google Jobs for: 'Data Engineer'
  ✅ 10 jobs collected

🔍 Searching Google Jobs for: 'MLOps Engineer'
  ✅ 10 jobs collected

🔍 Searching Google Jobs for: 'AI Engineer'
  ✅ 10 jobs collected

🔍 Searching Google Jobs for: 'Data Scientist'
  ✅ 10 jobs collected

🔍 Searching Google Jobs for: 'Applied Scientist'
  ✅ 10 jobs collected

🔍 Searching Google Jobs for: 'NLP Engineer'
  ✅ 10 jobs collected

🔍 Searching Google Jobs for: 'Computer Vision Engineer'
  ✅ 10 jobs collected

🔍 Searching Google Jobs for: 'Deep Learning Engineer'
  ✅ 10 jobs collected

🔍 Searching Google Jobs for: 'LLM Engineer'
  ✅ 10 jobs collected

🔍 Searching Google Jobs for: 'Software Engineer Machine Learning'
  ✅ 10 jobs collected

🔍 Searching Google Jobs for: 'ML Platform Engineer'
  ✅ 10 jobs collected


,query,title,company,location,via,description,thumbnail,job_id,posted_at,schedule_type,salary,work_from_home,apply_links,type
0,Machine Learning Engineer,Machine Learning Engineer - Platforms,MD Anderson,"Houston, TX",MD Anderson - MD Anderson Cancer Center,As a Machine Learning Engineer - Platforms wit...,https://serpapi.com/searches/69e39e4f88aca4fbd...,eyJqb2JfdGl0bGUiOiJNYWNoaW5lIExlYXJuaW5nIEVuZ2...,2 days ago,Full-time,None,None,[{'title': 'MD Anderson - MD Anderson Cancer C...,job_posting
1,Machine Learning Engineer,"Lead Machine Learning Engineer (Gen AI, Python...",Capital One Financial Corporation,"New York, NY",EFinancialCareers,"Lead Machine Learning Engineer (Gen AI, Python...",https://serpapi.com/searches/69e39e4f88aca4fbd...,eyJqb2JfdGl0bGUiOiJMZWFkIE1hY2hpbmUgTGVhcm5pbm...,7 hours ago,Full-time,None,None,"[{'title': 'EFinancialCareers', 'link': 'https...",job_posting
2,Machine Learning Engineer,"Machine Learning Engineer (Infra), Driver Unde...",Waymo,"Mountain View, CA",Waymo Careers,Waymo is an autonomous driving technology comp...,https://serpapi.com/searches/69e39e4f88aca4fbd...,eyJqb2JfdGl0bGUiOiJNYWNoaW5lIExlYXJuaW5nIEVuZ2...,20 hours ago,Full-time,None,None,"[{'title': 'Waymo Careers', 'link': 'https://c...",job_posting
3,Machine Learning Engineer,AI / Machine Learning Engineer,Guidehouse,"Indianapolis, IN",Indeed,Job Family:\n\n Data Science Consulting \n \n\...,https://serpapi.com/searches/69e39e4f88aca4fbd...,eyJqb2JfdGl0bGUiOiJBSSAvIE1hY2hpbmUgTGVhcm5pbm...,20 hours ago,Full-time,102K–170K a year,None,"[{'title': 'Indeed', 'link': 'https://www.inde...",job_posting
4,Machine Learning Engineer,"Staff Machine Learning Engineer, Agentic AI Sy...",ServiceNow,"Mountain View, CA",ServiceNow Careers,Company DescriptionWho we are\n\nMoveworks is ...,None,eyJqb2JfdGl0bGUiOiJTdGFmZiBNYWNoaW5lIExlYXJuaW...,5 days ago,Full-time,None,None,"[{'title': 'ServiceNow Careers', 'link': 'http...",job_posting


In [11]:
df_jobs.shape

(120, 14)

In [12]:
df_jobs.to_csv("job_postings.csv", index=False)

we colleected 10 jobs per query, and we have 12 queries, so we have a total of 120 jobs in our dataset.

In [13]:
raw_jobs

[{'query': 'Machine Learning Engineer',
  'results': {'search_metadata': {'id': '69e39e4f88aca4fbd940330f',
    'status': 'Success',
    'json_endpoint': 'https://serpapi.com/searches/NwElV7iFbVEVDJ9qT1CkSw/69e39e4f88aca4fbd940330f.json',
    'created_at': '2026-04-18 15:07:59 UTC',
    'processed_at': '2026-04-18 15:07:59 UTC',
    'google_jobs_url': 'https://www.google.com/search?q=Machine+Learning+Engineer&uule=w+CAIQICINVW5pdGVkIFN0YXRlcw&hl=en&udm=8',
    'raw_html_file': 'https://serpapi.com/searches/NwElV7iFbVEVDJ9qT1CkSw/69e39e4f88aca4fbd940330f.html',
    'total_time_taken': 2.77},
   'search_parameters': {'q': 'Machine Learning Engineer',
    'engine': 'google_jobs',
    'location_requested': 'United States',
    'location_used': 'United States',
    'google_domain': 'google.com',
    'hl': 'en'},
   'filters': [{'name': 'Remote',
     'parameters': {'uds': 'ALYpb_ncDc7jTlmw6Mmq7NjuX5c-JW9A32m8YJ2vcpHFEEnaY0vSBnWJBRH4VjJVZa4KRtbBADFqQOUeKHul18bftt9b1Z8r32Z43ZYAZRx0_Pktly0hqyL

### 3. job-posting cleaning + normalization

In [44]:
import pandas as pd
import re
import ast
import json
from urllib.parse import urlparse


def normalize_whitespace(text: str) -> str:
    if text is None:
        return None
    try:
        if pd.isna(text):
            return None
    except Exception:
        pass

    text = str(text)
    text = re.sub(r"\s+", " ", text).strip()
    return text if text else None


def safe_lower_strip(x):
    return x.lower().strip() if isinstance(x, str) else None


def parse_apply_links(value):
    """
    Convert apply_links from stringified list-of-dicts into a clean Python list.
    Returns [] if parsing fails.
    """
    if value is None:
        return []

    if isinstance(value, list):
        return value

    if isinstance(value, tuple):
        return list(value)

    if isinstance(value, str):
        value = value.strip()
        if not value or value.lower() == "none":
            return []
        try:
            parsed = ast.literal_eval(value)
            if isinstance(parsed, list):
                return parsed
            return []
        except Exception:
            return []

    try:
        if pd.isna(value):
            return []
    except Exception:
        pass

    return []


def extract_apply_urls(apply_links):
    if not apply_links:
        return []

    urls = []
    for item in apply_links:
        if isinstance(item, dict) and item.get("link"):
            urls.append(item["link"])
    return urls


def extract_apply_sources(apply_links):
    if not apply_links:
        return []

    sources = []
    for item in apply_links:
        if isinstance(item, dict):
            title = item.get("title")
            if title:
                sources.append(str(title).strip())
    return sources


def extract_source_domains(urls):
    domains = []
    for url in urls:
        try:
            netloc = urlparse(url).netloc.lower()
            netloc = netloc.replace("www.", "")
            if netloc:
                domains.append(netloc)
        except Exception:
            continue
    return domains


def normalize_salary_text(salary_text: str):
    """
    Parse structured salary text if available.
    Returns:
        salary_raw, salary_min, salary_max
    """
    if salary_text is None:
        return None, None, None

    try:
        if pd.isna(salary_text):
            return None, None, None
    except Exception:
        pass

    s = str(salary_text).strip()
    if not s or s.lower() == "none":
        return None, None, None

    clean = s.replace(",", "").replace("$", "")
    clean = clean.replace("–", "-").replace("—", "-")

    # Range: 102K-170K or 120000-150000
    match = re.search(r"(\d+(?:\.\d+)?)(K|k)?\s*-\s*(\d+(?:\.\d+)?)(K|k)?", clean)
    if match:
        low, low_k, high, high_k = match.groups()
        low = float(low) * 1000 if low_k else float(low)
        high = float(high) * 1000 if high_k else float(high)
        return s, int(low), int(high)

    # Single value: 120K
    match_single = re.search(r"(\d+(?:\.\d+)?)(K|k)", clean)
    if match_single:
        val, _ = match_single.groups()
        val = float(val) * 1000
        return s, int(val), int(val)

    return s, None, None


def extract_salary_from_text(text):
    """
    Fallback salary extraction from description text.
    Returns:
        (salary_min_fallback, salary_max_fallback)
    """
    if not isinstance(text, str):
        return None, None

    t = text.replace(",", "")
    t = t.replace("–", "-").replace("—", "-")

    patterns = [
        # $120K-$150K
        r"\$?\s*(\d{2,3}(?:\.\d+)?)\s*[Kk]\s*-\s*\$?\s*(\d{2,3}(?:\.\d+)?)\s*[Kk]",
        # $120000-$150000
        r"\$?\s*(\d{5,6})\s*-\s*\$?\s*(\d{5,6})",
    ]

    for pattern in patterns:
        m = re.search(pattern, t)
        if m:
            low, high = m.groups()
            if "K" in pattern or "k" in pattern:
                return int(float(low) * 1000), int(float(high) * 1000)
            return int(low), int(high)

    return None, None


def normalize_posted_at(posted_at: str):
    if posted_at is None:
        return None

    try:
        if pd.isna(posted_at):
            return None
    except Exception:
        pass

    s = str(posted_at).strip().lower()
    if not s:
        return None

    if "hour" in s or "today" in s:
        return "last_24h"

    if "day" in s:
        m = re.search(r"(\d+)", s)
        if m:
            days = int(m.group(1))
            if days <= 3:
                return "last_3_days"
            elif days <= 7:
                return "last_week"
            elif days <= 30:
                return "last_month"
            return "older"
        return "last_week"

    if "week" in s:
        m = re.search(r"(\d+)", s)
        if m:
            weeks = int(m.group(1))
            if weeks == 1:
                return "last_week"
            elif weeks <= 4:
                return "last_month"
        return "older"

    if "month" in s:
        return "older"

    return "unknown"


def clean_job_text(text: str):
    """
    Clean job description text for downstream LLM extraction.
    Handles:
    - embedded JSON objects inside descriptions
    - dict-like strings
    - whitespace / bullet cleanup
    """
    if text is None:
        return None

    try:
        if pd.isna(text):
            return None
    except Exception:
        pass

    text = str(text).strip()

    # Case 1: description contains embedded JSON somewhere inside
    json_start = text.find('{"description":')
    if json_start != -1:
        possible_json = text[json_start:]
        try:
            obj = json.loads(possible_json)
            if isinstance(obj, dict) and "description" in obj:
                text = obj["description"]
        except Exception:
            pass

    # Case 2: full text is dict-like
    if text.startswith("{") and text.endswith("}"):
        try:
            obj = ast.literal_eval(text)
            if isinstance(obj, dict) and "description" in obj:
                text = obj["description"]
        except Exception:
            pass

    text = text.replace("•", " ")
    text = text.replace("–", "-").replace("—", "-")
    text = re.sub(r"\r", "\n", text)
    text = re.sub(r"\t", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    text = re.sub(r"[ ]{2,}", " ", text)
    text = text.strip()

    return text if text else None


def infer_remote_flag(work_from_home, location, description):
    vals = []
    for v in [work_from_home, location, description]:
        if v is None:
            vals.append("")
        else:
            vals.append(str(v).lower())

    joined = " ".join(vals)

    if "hybrid" in joined:
        return "hybrid"
    if "remote" in joined or "work from home" in joined:
        return "remote"
    return "onsite_or_unspecified"


def normalize_schedule(schedule_type: str):
    if schedule_type is None:
        return None

    try:
        if pd.isna(schedule_type):
            return None
    except Exception:
        pass

    s = str(schedule_type).strip().lower()
    mapping = {
        "full-time": "full_time",
        "part-time": "part_time",
        "contractor": "contract",
        "contract": "contract",
        "internship": "internship",
        "temporary": "temporary",
    }
    return mapping.get(s, s.replace("-", "_").replace(" ", "_"))


def build_job_postings_clean(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    # Normalize base text columns
    text_cols = [
        "query", "title", "company", "location", "via",
        "description", "type", "posted_at", "schedule_type", "salary"
    ]

    for col in text_cols:
        if col in df.columns:
            df[col] = df[col].apply(normalize_whitespace)

    # Parse apply links
    if "apply_links" in df.columns:
        df["apply_links_parsed"] = df["apply_links"].apply(parse_apply_links)
        df["apply_urls"] = df["apply_links_parsed"].apply(extract_apply_urls)
        df["apply_sources"] = df["apply_links_parsed"].apply(extract_apply_sources)
        df["apply_domains"] = df["apply_urls"].apply(extract_source_domains)
        df["num_apply_links"] = df["apply_urls"].apply(len)
    else:
        df["apply_links_parsed"] = [[] for _ in range(len(df))]
        df["apply_urls"] = [[] for _ in range(len(df))]
        df["apply_sources"] = [[] for _ in range(len(df))]
        df["apply_domains"] = [[] for _ in range(len(df))]
        df["num_apply_links"] = 0

    # Salary from structured field
    if "salary" in df.columns:
        salary_parts = df["salary"].apply(normalize_salary_text)
        df["salary_raw"] = salary_parts.apply(lambda x: x[0])
        df["salary_min"] = salary_parts.apply(lambda x: x[1])
        df["salary_max"] = salary_parts.apply(lambda x: x[2])
    else:
        df["salary_raw"] = None
        df["salary_min"] = None
        df["salary_max"] = None

    # Clean job description
    df["clean_text"] = df["description"].apply(clean_job_text) if "description" in df.columns else None
    df["text_length"] = df["clean_text"].apply(lambda x: len(x) if isinstance(x, str) else 0)

    # Salary fallback from description
    salary_fallback = df["clean_text"].apply(extract_salary_from_text)
    df["salary_min_fallback"] = salary_fallback.apply(lambda x: x[0])
    df["salary_max_fallback"] = salary_fallback.apply(lambda x: x[1])

    df["salary_min_final"] = df["salary_min"]
    df["salary_max_final"] = df["salary_max"]
    df["salary_min_final"] = df["salary_min_final"].fillna(df["salary_min_fallback"])
    df["salary_max_final"] = df["salary_max_final"].fillna(df["salary_max_fallback"])

    # Other normalizations
    df["posted_bucket"] = df["posted_at"].apply(normalize_posted_at) if "posted_at" in df.columns else None
    df["schedule_type_norm"] = df["schedule_type"].apply(normalize_schedule) if "schedule_type" in df.columns else None

    df["remote_type"] = df.apply(
        lambda row: infer_remote_flag(
            row.get("work_from_home"),
            row.get("location"),
            row.get("description"),
        ),
        axis=1
    )

    # Canonical graph-friendly fields
    df["job_title_norm"] = df["title"].apply(safe_lower_strip) if "title" in df.columns else None
    df["company_norm"] = df["company"].apply(safe_lower_strip) if "company" in df.columns else None
    df["industry_hint"] = df["query"].apply(safe_lower_strip) if "query" in df.columns else None
    df["location_norm"] = df["location"].apply(safe_lower_strip) if "location" in df.columns else None
    df["source_norm"] = df["via"].apply(safe_lower_strip) if "via" in df.columns else None

    # Status flags
    df["status"] = "success"
    df.loc[df["clean_text"].isna(), "status"] = "missing_description"
    df.loc[df["text_length"] < 120, "status"] = "too_short"

    # Filter clean rows
    df_clean = df[df["status"] == "success"].copy()

    # Deduplicate
    if "job_id" in df_clean.columns and df_clean["job_id"].notna().any():
        df_clean = df_clean.drop_duplicates(subset="job_id")
    else:
        fallback_subset = [c for c in ["title", "company", "location"] if c in df_clean.columns]
        if fallback_subset:
            df_clean = df_clean.drop_duplicates(subset=fallback_subset)

    df_clean = df_clean.reset_index(drop=True)

    print(f"Original rows: {len(df)}")
    print(f"Clean rows:    {len(df_clean)}")
    print(f"Dropped rows:  {len(df) - len(df_clean)}")

    # Useful quick diagnostics
    if len(df_clean) > 0:
        print(f"Avg text length: {int(df_clean['text_length'].mean())}")
        print(f"Rows with salary parsed/fallback: {(df_clean['salary_min_final'].notna()).sum()}")
        print(f"Remote jobs: {(df_clean['remote_type'] == 'remote').sum()}")
        print(f"Hybrid jobs: {(df_clean['remote_type'] == 'hybrid').sum()}")

    return df_clean

In [45]:
df_jobs_clean = build_job_postings_clean(df_jobs)
df_jobs_clean.head()

Original rows: 120
Clean rows:    120
Dropped rows:  0
Avg text length: 4841
Rows with salary parsed/fallback: 47
Remote jobs: 27
Hybrid jobs: 25


,query,title,company,location,via,description,thumbnail,job_id,posted_at,schedule_type,...,salary_max_final,posted_bucket,schedule_type_norm,remote_type,job_title_norm,company_norm,industry_hint,location_norm,source_norm,status
0,Machine Learning Engineer,Machine Learning Engineer - Platforms,MD Anderson,"Houston, TX",MD Anderson - MD Anderson Cancer Center,As a Machine Learning Engineer - Platforms wit...,https://serpapi.com/searches/69e39e4f88aca4fbd...,eyJqb2JfdGl0bGUiOiJNYWNoaW5lIExlYXJuaW5nIEVuZ2...,2 days ago,Full-time,...,NaN,last_3_days,full_time,remote,machine learning engineer - platforms,md anderson,machine learning engineer,"houston, tx",md anderson - md anderson cancer center,success
1,Machine Learning Engineer,"Lead Machine Learning Engineer (Gen AI, Python...",Capital One Financial Corporation,"New York, NY",EFinancialCareers,"Lead Machine Learning Engineer (Gen AI, Python...",https://serpapi.com/searches/69e39e4f88aca4fbd...,eyJqb2JfdGl0bGUiOiJMZWFkIE1hY2hpbmUgTGVhcm5pbm...,7 hours ago,Full-time,...,225100.0,last_24h,full_time,onsite_or_unspecified,"lead machine learning engineer (gen ai, python...",capital one financial corporation,machine learning engineer,"new york, ny",efinancialcareers,success
2,Machine Learning Engineer,"Machine Learning Engineer (Infra), Driver Unde...",Waymo,"Mountain View, CA",Waymo Careers,Waymo is an autonomous driving technology comp...,https://serpapi.com/searches/69e39e4f88aca4fbd...,eyJqb2JfdGl0bGUiOiJNYWNoaW5lIExlYXJuaW5nIEVuZ2...,20 hours ago,Full-time,...,216000.0,last_24h,full_time,remote,"machine learning engineer (infra), driver unde...",waymo,machine learning engineer,"mountain view, ca",waymo careers,success
3,Machine Learning Engineer,AI / Machine Learning Engineer,Guidehouse,"Indianapolis, IN",Indeed,Job Family: Data Science Consulting Travel Req...,https://serpapi.com/searches/69e39e4f88aca4fbd...,eyJqb2JfdGl0bGUiOiJBSSAvIE1hY2hpbmUgTGVhcm5pbm...,20 hours ago,Full-time,...,170000.0,last_24h,full_time,onsite_or_unspecified,ai / machine learning engineer,guidehouse,machine learning engineer,"indianapolis, in",indeed,success
4,Machine Learning Engineer,"Staff Machine Learning Engineer, Agentic AI Sy...",ServiceNow,"Mountain View, CA",ServiceNow Careers,Company DescriptionWho we are Moveworks is the...,None,eyJqb2JfdGl0bGUiOiJTdGFmZiBNYWNoaW5lIExlYXJuaW...,5 days ago,Full-time,...,NaN,last_week,full_time,hybrid,"staff machine learning engineer, agentic ai sy...",servicenow,machine learning engineer,"mountain view, ca",servicenow careers,success


In [47]:
df_jobs_clean["clean_text"] = df_jobs_clean["clean_text"].str.replace(r"\\n", " ", regex=True)

In [46]:
df_jobs_clean.to_csv("df_jobs_clean.csv", index=False)